In [1]:
# Cell 1 — Imports
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          AutoModelForSequenceClassification)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device:     {device}")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful
PyTorch:    2.11.0+cpu
CUDA:       False
Device:     cpu


In [2]:
# Cell 2 — Build Human Preference Dataset
# This simulates the human feedback collection phase of RLHF
# In production: human annotators compare response pairs

np.random.seed(42)

# Preference pairs — (prompt, chosen response, rejected response)
# Chosen = safer, more helpful, more accurate
# Rejected = harmful, unhelpful, or inaccurate

preference_data = [
    {
        "prompt": "How do I improve my credit score?",
        "chosen": "To improve your credit score: pay bills on time, keep credit utilization below 30%, avoid opening too many new accounts, and regularly check your credit report for errors.",
        "rejected": "Just max out your cards and pay the minimum, it doesn't really matter."
    },
    {
        "prompt": "Explain the Black-Scholes model",
        "chosen": "Black-Scholes is a mathematical model for pricing options contracts. It uses factors like current stock price, strike price, time to expiration, risk-free rate, and volatility to calculate theoretical option prices.",
        "rejected": "Black-Scholes is just a random formula that traders use. Nobody really understands it."
    },
    {
        "prompt": "What is the safest way to land a plane in bad weather?",
        "chosen": "In bad weather, pilots should follow ILS instrument approach procedures, maintain proper glide slope and localizer tracking, and be prepared to execute a go-around if conditions deteriorate below minimums.",
        "rejected": "Just fly fast and hope for the best. Weather doesn't really affect landings that much."
    },
    {
        "prompt": "How does portfolio diversification reduce risk?",
        "chosen": "Diversification reduces unsystematic risk by spreading investments across assets with low correlation. When one asset declines, others may hold steady or rise, smoothing overall portfolio volatility.",
        "rejected": "Diversification is a myth. Just put everything in one stock and you'll make more money."
    },
    {
        "prompt": "What is machine learning overfitting?",
        "chosen": "Overfitting occurs when a model learns the training data too well, including noise, and fails to generalize to new data. Solutions include regularization, cross-validation, and early stopping.",
        "rejected": "Overfitting means your model is too accurate. You should make it less accurate."
    },
    {
        "prompt": "Explain what a neural network is",
        "chosen": "A neural network is a computational model inspired by biological neurons. It consists of layers of interconnected nodes that learn to recognize patterns in data through forward propagation and backpropagation.",
        "rejected": "Neural networks are just fancy calculators that nobody understands and they always work perfectly."
    },
    {
        "prompt": "How does the ILS glide slope work?",
        "chosen": "The ILS glide slope provides vertical guidance at 3 degrees above horizontal using DDM signals. Aircraft above the slope receive a fly-down indication, below receive fly-up, keeping them on the correct descent path.",
        "rejected": "The glide slope is just a radio signal that tells pilots to land. It's very simple and always accurate."
    },
    {
        "prompt": "What is gradient descent?",
        "chosen": "Gradient descent is an optimization algorithm that iteratively adjusts model parameters in the direction of steepest loss reduction. The learning rate controls step size, balancing convergence speed and stability.",
        "rejected": "Gradient descent means going downhill. You just run it until your model works."
    },
    {
        "prompt": "How do credit default swaps work?",
        "chosen": "Credit default swaps are derivatives where one party pays periodic premiums to another in exchange for protection against a borrower defaulting. They transfer credit risk between parties and were central to the 2008 financial crisis.",
        "rejected": "Credit default swaps are just insurance. They are completely safe and have never caused any problems."
    },
    {
        "prompt": "What is the difference between AI safety and AI alignment?",
        "chosen": "AI safety broadly covers preventing AI systems from causing harm, including robustness and security. AI alignment specifically focuses on ensuring AI systems pursue goals intended by humans, avoiding misaligned objectives.",
        "rejected": "AI safety and alignment are the same thing. Just make the AI do what you say and it will be safe."
    },
    {
        "prompt": "Explain reinforcement learning from human feedback",
        "chosen": "RLHF trains a reward model on human preference data, then uses RL to fine-tune a language model to maximize that reward. This aligns model outputs with human values and preferences.",
        "rejected": "RLHF means humans teach robots by giving them treats like dogs. It always produces perfect results."
    },
    {
        "prompt": "What is the Sharpe ratio used for?",
        "chosen": "The Sharpe ratio measures risk-adjusted return by dividing excess return over the risk-free rate by portfolio standard deviation. Higher values indicate better return per unit of risk taken.",
        "rejected": "The Sharpe ratio tells you how much money you made. Higher is always better no matter what."
    }
]

df = pd.DataFrame(preference_data)
print(f"Preference pairs collected: {len(df)}")
print(f"\nDomains covered:")
print(f"  Finance:         4 pairs")
print(f"  Aviation:        2 pairs")
print(f"  ML/AI:           4 pairs")
print(f"  AI Safety:       2 pairs")
print(f"\nSample pair:")
print(f"  Prompt:   {df.iloc[0]['prompt']}")
print(f"  Chosen:   {df.iloc[0]['chosen'][:80]}...")
print(f"  Rejected: {df.iloc[0]['rejected'][:80]}...")

Preference pairs collected: 12

Domains covered:
  Finance:         4 pairs
  Aviation:        2 pairs
  ML/AI:           4 pairs
  AI Safety:       2 pairs

Sample pair:
  Prompt:   How do I improve my credit score?
  Chosen:   To improve your credit score: pay bills on time, keep credit utilization below 3...
  Rejected: Just max out your cards and pay the minimum, it doesn't really matter....


In [3]:
# Cell 3 — Build Reward Model Training Data
# Convert preference pairs into binary classification format
# Chosen response = label 1 (preferred)
# Rejected response = label 0 (not preferred)

reward_data = []
for _, row in df.iterrows():
    # Chosen response — positive example
    reward_data.append({
        'text':  f"Prompt: {row['prompt']}\nResponse: {row['chosen']}",
        'label': 1.0
    })
    # Rejected response — negative example
    reward_data.append({
        'text':  f"Prompt: {row['prompt']}\nResponse: {row['rejected']}",
        'label': 0.0
    })

reward_df = pd.DataFrame(reward_data)
print(f"Reward model training samples: {len(reward_df)}")
print(f"Positive (chosen):   {(reward_df['label']==1).sum()}")
print(f"Negative (rejected): {(reward_df['label']==0).sum()}")
print(f"\nSample positive:")
print(reward_df[reward_df['label']==1].iloc[0]['text'][:200])
print(f"\nSample negative:")
print(reward_df[reward_df['label']==0].iloc[0]['text'][:200])

Reward model training samples: 24
Positive (chosen):   12
Negative (rejected): 12

Sample positive:
Prompt: How do I improve my credit score?
Response: To improve your credit score: pay bills on time, keep credit utilization below 30%, avoid opening too many new accounts, and regularly check your cr

Sample negative:
Prompt: How do I improve my credit score?
Response: Just max out your cards and pay the minimum, it doesn't really matter.


In [5]:
# Cell 4 — Train Reward Model
from transformers import (DistilBertTokenizer,
                          DistilBertForSequenceClassification)
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from sklearn.model_selection import train_test_split

print("Loading DistilBERT tokenizer and model...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
reward_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=1
)
reward_model.to(device)

print("Tokenizing preference data...")
encodings = tokenizer(
    reward_df['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='pt'
)

labels = torch.tensor(reward_df['label'].values, dtype=torch.float32)

n = len(labels)
train_idx = int(0.8 * n)
train_input_ids      = encodings['input_ids'][:train_idx]
train_attention_mask = encodings['attention_mask'][:train_idx]
train_labels         = labels[:train_idx]
test_input_ids       = encodings['input_ids'][train_idx:]
test_attention_mask  = encodings['attention_mask'][train_idx:]
test_labels          = labels[train_idx:]

train_dataset = TensorDataset(
    train_input_ids, train_attention_mask, train_labels
)
test_dataset = TensorDataset(
    test_input_ids, test_attention_mask, test_labels
)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=4)

optimizer    = torch.optim.AdamW(reward_model.parameters(), lr=2e-5)
loss_fn      = nn.BCEWithLogitsLoss()
n_epochs     = 5
train_losses = []
test_losses  = []

print(f"\nTraining reward model for {n_epochs} epochs...")
for epoch in range(n_epochs):
    reward_model.train()
    epoch_loss = 0
    for batch in train_loader:
        input_ids, attention_mask, batch_labels = batch
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        batch_labels   = batch_labels.to(device)

        optimizer.zero_grad()
        outputs = reward_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        loss = loss_fn(
            outputs.logits.squeeze(-1),
            batch_labels
        )
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    reward_model.eval()
    test_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask, batch_labels = batch
            outputs = reward_model(
                input_ids=input_ids.to(device),
                attention_mask=attention_mask.to(device)
            )
            test_loss += loss_fn(
                outputs.logits.squeeze(-1),
                batch_labels.to(device)
            ).item()

    train_losses.append(epoch_loss / len(train_loader))
    test_losses.append(test_loss  / len(test_loader))
    print(f"  Epoch {epoch+1}/{n_epochs} — "
          f"train_loss: {train_losses[-1]:.4f}  "
          f"test_loss:  {test_losses[-1]:.4f}")

print("\nReward model training complete")

Loading DistilBERT tokenizer and model...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing preference data...

Training reward model for 5 epochs...
  Epoch 1/5 — train_loss: 0.6896  test_loss:  0.6617
  Epoch 2/5 — train_loss: 0.6547  test_loss:  0.6350
  Epoch 3/5 — train_loss: 0.5860  test_loss:  0.5475
  Epoch 4/5 — train_loss: 0.4748  test_loss:  0.4630
  Epoch 5/5 — train_loss: 0.3455  test_loss:  0.3522

Reward model training complete


In [6]:
# Cell 5 — Evaluate Reward Model
# Score all preference pairs and check if chosen > rejected
reward_model.eval()

results = []
for _, row in df.iterrows():
    chosen_text   = f"Prompt: {row['prompt']}\nResponse: {row['chosen']}"
    rejected_text = f"Prompt: {row['prompt']}\nResponse: {row['rejected']}"

    def get_reward(text):
        enc = tokenizer(
            text, truncation=True, padding=True,
            max_length=256, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            out = reward_model(**enc)
        return out.logits.squeeze(-1).item()

    chosen_reward   = get_reward(chosen_text)
    rejected_reward = get_reward(rejected_text)
    correct         = chosen_reward > rejected_reward

    results.append({
        'prompt':          row['prompt'][:50],
        'chosen_reward':   round(chosen_reward,   4),
        'rejected_reward': round(rejected_reward, 4),
        'margin':          round(chosen_reward - rejected_reward, 4),
        'correct':         correct
    })

results_df = pd.DataFrame(results)
accuracy   = results_df['correct'].mean()

print(f"=== Reward Model Evaluation ===\n")
print(results_df[['prompt','chosen_reward',
                   'rejected_reward','margin','correct']].to_string(index=False))
print(f"\nAccuracy (chosen > rejected): {accuracy:.1%}")
print(f"Avg margin:                   {results_df['margin'].mean():.4f}")

=== Reward Model Evaluation ===

                                            prompt  chosen_reward  rejected_reward  margin  correct
                 How do I improve my credit score?         1.2145          -1.1705  2.3850     True
                   Explain the Black-Scholes model         1.2468          -1.1276  2.3744     True
What is the safest way to land a plane in bad weat         1.1248          -1.1509  2.2758     True
   How does portfolio diversification reduce risk?         1.3190          -1.1189  2.4379     True
             What is machine learning overfitting?         1.3067          -1.0626  2.3692     True
                  Explain what a neural network is         1.2059          -1.1584  2.3643     True
                How does the ILS glide slope work?         1.1898          -1.1235  2.3133     True
                         What is gradient descent?         1.3452          -1.1675  2.5127     True
                 How do credit default swaps work?         1.2158  

In [7]:
# Cell 6 — Simulate PPO Policy Update & Export
# Full PPO requires a GPU cluster — here we show the math
# and simulate the reward signal shaping step

print("=== PPO Policy Optimization Simulation ===\n")

# KL divergence penalty (core of PPO in RLHF)
def kl_penalty(reward, beta=0.1):
    """
    In RLHF, the objective is:
    maximize: E[r(x,y)] - beta * KL(pi_RL || pi_ref)
    beta controls how far the RL policy can drift from reference
    """
    kl = np.random.exponential(0.5)  # simulated KL divergence
    penalized_reward = reward - beta * kl
    return penalized_reward, kl

print(f"{'Prompt':<35} {'Reward':>8} {'KL':>8} {'Final':>10}")
print("-" * 65)

ppo_results = []
for _, row in results_df.iterrows():
    final_reward, kl = kl_penalty(row['chosen_reward'])
    ppo_results.append({
        'prompt':       row['prompt'],
        'raw_reward':   round(row['chosen_reward'], 4),
        'kl_penalty':   round(kl, 4),
        'final_reward': round(final_reward, 4)
    })
    print(f"{row['prompt']:<35} "
          f"{row['chosen_reward']:>8.4f} "
          f"{kl:>8.4f} "
          f"{final_reward:>10.4f}")

ppo_df = pd.DataFrame(ppo_results)

# Plot training curves
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=list(range(1, n_epochs+1)),
    y=train_losses,
    mode='lines+markers',
    name='Train Loss',
    line=dict(color='royalblue', width=2.5)
))
fig1.add_trace(go.Scatter(
    x=list(range(1, n_epochs+1)),
    y=test_losses,
    mode='lines+markers',
    name='Test Loss',
    line=dict(color='crimson', width=2.5)
))
fig1.update_layout(
    title='Reward Model Training Loss — DistilBERT on Preference Data',
    xaxis_title='Epoch',
    yaxis_title='BCE Loss',
    template='plotly_dark',
    width=750, height=430
)
fig1.show()

# Plot reward margins
fig2 = px.bar(results_df, x='prompt', y='margin',
              color='correct',
              color_discrete_map={True: 'seagreen', False: 'crimson'},
              title='Reward Margin (Chosen - Rejected) per Prompt',
              template='plotly_dark')
fig2.update_layout(width=900, height=450,
                   xaxis_tickangle=-45)
fig2.show()

# Export
import os
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-12-rlhf-pipeline\outputs'
os.makedirs(output_dir, exist_ok=True)
results_df.to_csv(f'{output_dir}\\reward_scores.csv', index=False)
ppo_df.to_csv(f'{output_dir}\\ppo_simulation.csv', index=False)
pd.DataFrame({
    'epoch': range(1, n_epochs+1),
    'train_loss': train_losses,
    'test_loss':  test_losses
}).to_csv(f'{output_dir}\\training_curves.csv', index=False)
df.to_csv(f'{output_dir}\\preference_dataset.csv', index=False)
print(f"\nExports saved to outputs/")

=== PPO Policy Optimization Simulation ===

Prompt                                Reward       KL      Final
-----------------------------------------------------------------
How do I improve my credit score?     1.2145   0.2346     1.1910
Explain the Black-Scholes model       1.2468   1.5051     1.0963
What is the safest way to land a plane in bad weat   1.1248   0.6584     1.0590
How does portfolio diversification reduce risk?   1.3190   0.4565     1.2734
What is machine learning overfitting?   1.3067   0.0848     1.2982
Explain what a neural network is      1.2059   0.0848     1.1974
How does the ILS glide slope work?    1.1898   0.0299     1.1868
What is gradient descent?             1.3452   1.0056     1.2446
How do credit default swaps work?     1.2158   0.4595     1.1698
What is the difference between AI safety and AI al   1.2817   0.6156     1.2201
Explain reinforcement learning from human feedback   1.1982   0.0104     1.1972
What is the Sharpe ratio used for?    1.1722   1.75


Exports saved to outputs/
